# Alpha AML — Fraud Anomaly Detection (exploration)

This notebook prototypes the ML risk layer that the `ml_score` Airflow DAG runs in
production (`src/ml/train.py`). It loads the 30-day per-customer behavioral feature
frame, fits an **Isolation Forest** for unsupervised anomaly detection, and trains a
**supervised triage** model (Gradient Boosting) against the rule-engine alert label,
benchmarking it with ROC-AUC / PR-AUC and feature importances.

> Requires the platform's Postgres reachable via the `POSTGRES_*` env vars and
> `scikit-learn`, `matplotlib` installed in the kernel.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # make `src` importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ml.features import FEATURE_COLUMNS, build_matrix, get_engine, load_features

df = load_features(get_engine())
print(f'{len(df):,} active customers, {df["rule_flagged"].sum():,} rule-flagged')
df.head()

## 1. Unsupervised anomaly detection — Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest

X = build_matrix(df)
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X)

raw = -iso.decision_function(X)
df['anomaly_score'] = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
df['is_anomaly'] = iso.predict(X) == -1

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df.loc[~df.rule_flagged, 'anomaly_score'], bins=40, alpha=0.7, label='not flagged')
ax.hist(df.loc[df.rule_flagged, 'anomaly_score'], bins=40, alpha=0.7, label='rule-flagged')
ax.set_xlabel('anomaly score'); ax.set_ylabel('customers'); ax.legend(); plt.show()

df.sort_values('anomaly_score', ascending=False).head(10)[
    ['customer_id', 'anomaly_score', 'rule_flagged', 'txn_count_30d', 'volume_30d']
]

## 2. Supervised triage — Gradient Boosting vs the rule baseline

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

y = df['rule_flagged'].astype(int).to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = GradientBoostingClassifier(random_state=42).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

print(f'ROC-AUC = {roc_auc_score(y_te, proba):.3f}   PR-AUC = {average_precision_score(y_te, proba):.3f}')

fpr, tpr, _ = roc_curve(y_te, proba)
prec, rec, _ = precision_recall_curve(y_te, proba)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], '--', c='grey')
axes[0].set_title('ROC'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[1].plot(rec, prec, c='crimson'); axes[1].set_title('Precision-Recall')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); plt.show()

In [ ]:
imp = pd.Series(clf.feature_importances_, index=FEATURE_COLUMNS).sort_values()
imp.plot.barh(figsize=(7, 4), title='Feature importance'); plt.show()

## 3. Rule engine vs ML anomaly — overlap

Where do the deterministic rules and the unsupervised model agree? `ML only` are
behavioral outliers the static thresholds missed; `Rule only` are policy hits the
model did not consider anomalous.

In [ ]:
pd.crosstab(df.rule_flagged, df.is_anomaly,
            rownames=['rule_flagged'], colnames=['ml_anomaly'])